# Tests on the TrustRegionRadius.jl package

📖 **Documentation:** <https://jeremyrieussec.github.io/TrustRegionRadius.jl/dev/>

To use the package TrustRegionRadius in development mode, uncomment the following line and run it once
```julia
Pkg.develop(path="C:/Users/jerem/OneDrive/Desktop/GitHub/TrustRegionRadius.jl") 
```
Packages used in this notebook:
```julia 
Pkg.add(["ADNLPModels", "CUTEst", "LinearAlgebra",  "Plots", "StatsPlots", "TOML", "Dates", "Printf", "NLPModels", "JLD2"])
``` 

In [1]:
using Pkg
Pkg.activate(".")

include("initialisation.jl")

  Activating project at `c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark`


| Experiment | Julia command | Description |
|------------|---------------|-------------|
| **1** | `comparison()` | Comparison of different trust-region radius update rules. |
| **2** | `trajectories()` | Comparison of trajectories for the gradient norm, trust-region radius, and trust-region ratio for different trust-region radius update rules. |
| **3** | `zeta_sweep()` | Comparison of different $\zeta$ values for R-DFO. |
| **4** | `mu_sweep()` | Comparison of different $\mu_{\max}$ values for R-Grad. |
| **5** | `inactivity()` | Direct measurement of trust-region inactivity. |
| **6** | `interaction()` | Interaction between the trust-region radius update mechanism and the model Hessian. |
| **7** | `CVRate()` | Comparison of the observed local convergence order. |
| **8** | `single_problem_experiment(::String)` | Runs all mechanisms on a single problem with detailed per-iteration diagnostics. ⚠️ **source file missing** — see note below. |
| **9** | `second_order()` | First- versus second-order anchoring: $\tau = \max\{\|g\|, -\lambda_{\min}\}$ against $\|g\|$ at a saddle. |
| **11** | `bhhh_study()` | Outer-product Hessians: where BHHH is justified and where it is not. |
| **12** | `sampling_examples()` | The three worked examples under every sampling rule. |

⚠️ **Two gaps in the suite.** The experiment filenames were shifted by one against their
contents (`exp2_trajectories.jl` defined `comparison()`, i.e. experiment 1, and so on down
the line). They have been renamed so each file matches what it defines — which leaves
**experiment 8 with no source at all**, since the file that carried its name held
experiment 7. `single_problem_experiment` is undefined until that file is written, and cell
11 below will fail. **Experiment 10** is absent as well: the suite jumps from 9 to 11.

In [2]:
# =============================================================================
# Central configuration — mirrors benchmark/config.jl.
#
# `initialisation.jl` already includes config.jl, so re-running this cell is only
# needed if you want to override something interactively. Re-running it with a
# DIFFERENT value will warn about redefining a constant and keep the old one;
# restart the kernel to change a `const`.
# =============================================================================

# Solver parameters — identical for every mechanism, deliberately.
# Per-rule tuning would measure tuning effort rather than algorithmic merit.
SOLVER_PARAMS   # tol = 1e-5, η = η1 = 0.1, η2 = 0.9, Δ0 = 1.0

# The same, with the second-order stopping test switched on. tol_H = -1 in
# SOLVER_PARAMS disables it and costs no curvature estimate; tol_H > 0 makes the
# run refuse to stop at a saddle and report :second_order when it certifies
# λ_min(B) ≥ -tol_H.
SECOND_ORDER_PARAMS

# The mechanisms. Note the keywords are ASCII (γ1, γ2, γ3, η1, η2, Δ0); the
# subscript spellings are accepted as aliases.
for (nm, f) in RULES
    println(rpad(nm, 20), f())
end
println()
println("τ-anchored twins (experiment 9):")
for (nm, f) in TAU_RULES
    println(rpad(nm, 22), asymptotic_regime(f()))
end

RDelta              RDelta(γ1 = 0.25, γ2 = 0.5, γ3 = 2.0, Δmin = 0.0, Δmax = Inf)
RStep               RStep(γ1 = 0.25, γ2 = 0.8, γ3 = 2.0, Δmin = 1.0e-14, Δmax = Inf, contract_on_step = true)
RDFO                RDFO(γ1 = 0.25, γ2 = 0.5, γ3 = 2.0, ζ = 100.0, Δmin = 0.0, Δmax = Inf)
RGrad               RGrad(γ1 = 0.25, γ2 = 0.5, γ3 = 2.0, μ = 1.0, μ0 = 1.0, half_test = true, Δmin = 0.0, Δmax = Inf)
RGradCapped         RGradCapped(γ1 = 0.25, γ2 = 0.5, γ3 = 2.0, μ = 1.0, μ0 = 1.0, μ_max = 128.0, half_test = true, Δmin = 0.0, Δmax = Inf)
RAdaptiveStep       RAdaptiveStep(γ1 = 0.0625, γ2 = 0.5, γ3 = 4.0, λ1 = 5.0, λ2 = 5.0, Δmin = 1.0e-14, Δmax = Inf)
RAdaptiveGrad       RAdaptiveGrad(μ = 1.0, μ0 = 1.0, γ1 = 0.0625, γ2 = 0.5, γ3 = 4.0, λ1 = 5.0, λ2 = 5.0, Δmin = 0.0, Δmax = Inf)

τ-anchored twins (experiment 9):
RGradTau              vanishing
RGradCappedTau        vanishing
RDFOTau               vanishing
RAdaptiveGradTau      vanishing


In [ ]:
comparison() 

In [ ]:
trajectories()

In [ ]:
zeta_sweep()

In [ ]:
mu_sweep()

In [ ]:
inactivity()

In [ ]:
interaction()

In [ ]:
CVRate()

---
## Experiment 9 — first- versus second-order anchoring

Every mechanism controls the radius against a *criticality measure*. The first-order
measure $\|g_k\|$ vanishes at **every** critical point — minimisers, saddles and maxima
alike. The second-order measure

$$\tau_k = \max\bigl\{\|g_k\|,\; -\lambda_{\min}(B_k)\bigr\}$$

vanishes only where the model is genuinely second-order critical.

The test problem is $f = x^4/4 - x^2/2 + y^2/2$, started at $(0, 0.7)$. The $x$-axis is
invariant under the gradient flow, so the trajectory runs **exactly** into the saddle at
the origin — the failure needs no tuning to provoke.

Four claims, and the last two are the ones worth internalising:

1. **The anchor collapses.** `RGrad` sets $\Delta = \mu\|g\| \to 0$ and halts; `RDFO`
   contracts geometrically for ever. Both then report `:first_order` — correctly by their
   own test. It is the test that is wrong.
2. **The measure gap.** $\tau_k/\|g_k\|$ is 1 on the convex part and unbounded at the
   saddle.
3. **The subsolver is half the story.** $\tau$ keeps the radius positive; it does not make
   the step go anywhere. Only $\tau$ *and* `EigenPoint` together reach a minimiser.
4. **The model is the other half.** Over `LBFGSModel`, $\lambda_{\min} > 0$ always, so
   $\tau \equiv \|g\|$ and the whole apparatus is an expensive no-op — which will still
   report `:second_order` at a saddle. The status certifies the *model*, not the function.

In [ ]:
second_order()

### Reading `exp9_paired.txt`

The paired sweep runs each $\|g\|$-anchored rule beside its $\tau$-twin through the shared
harness, so the runs are archived and resumable like every other experiment.

Read the **certified `:second_order`** column against **solved**. A configuration that
solves every problem while certifying none has been running a first-order method under a
second-order name — which is exactly what a $\tau$-rule over a positive semidefinite model
does.

Two things had to change in the harness before this experiment could report anything at
all, and both are worth knowing about if you have archived runs from before:

- **`solved` now counts `:second_order`.** It tested `=== :first_order`, so a $\tau$-run
  that certified a minimiser was recorded as a *failure*: the columns doing best would
  have shown zero reliability.
- **`RunRecord` now carries `tau_traj` and `lambda_traj`.** They were dropped, so claim 2
  could not be plotted from an archived run.

Archives written before those fields existed reload fine — the missing keys default rather
than raising — but their `tau_traj` will be empty.

In [ ]:
# The measure gap on a single run, without the archive layer.
st = tr_solve(ADNLPModel(v -> v[1]^4/4 - v[1]^2/2 + v[2]^2/2, [0.0, 0.7]);
              rule      = RGradTau(μ = 1.0),
              model     = ExactHessian(),              # must report λ_min < 0
              subsolver = EigenPoint(SteihaugCG()),    # must exploit it
              params    = TRParams(tol = 1e-8, tol_H = 1e-6), trace = true)

ss = st.solver_specific
gap = ss[:tau_trajectory] ./ max.(ss[:grad_trajectory], 1e-300)

println("status        : ", st.status)
println("solution      : ", round.(st.solution, digits = 6), "   (saddle is [0,0])")
println("max τ/‖g‖     : ", round(maximum(gap), digits = 3))
println("min λ_min(B)  : ", round(minimum(ss[:lambda_min_trajectory]), digits = 4))

plot(0:length(gap)-1, gap; yscale = :log10, lw = 2, legend = :topleft,
     xlabel = "iteration k", ylabel = "τ_k / ‖g_k‖", label = "RGradTau",
     title = "where the gradient calls the point critical and the curvature does not")
hline!([1.0]; ls = :dot, c = :black, label = "τ = ‖g‖")

In [ ]:
# ⚠️ Experiment 8 has no source file (see the note under the table above), so
# `single_problem_experiment` is undefined. The CUTEst setup is kept so the cell
# is ready the moment that file exists.
try
    finalize(my_nlp)
catch e
    @warn "No need to finalize: $e"
end

my_nlp = CUTEstModel("ROSENBR")

println("#"^70)
println("Tracing problem: $(my_nlp.meta.name), with : \n  + $(my_nlp.meta.nvar) variables \n  + x0 = $(my_nlp.meta.x0).")
println("#"^70)

if isdefined(Main, :single_problem_experiment)
    single_problem_experiment("ROSENBR")
else
    @warn "single_problem_experiment is not defined: experiments/exp8_single_problem.jl is missing."
end

---
## Experiments 11 and 12 — the sampled problem classes

These exercise the fourth axis. Both were adapted to the class split:

- `SampledNLP` → **`FiniteSumNLP`**, since every example here is a finite sum. The
  expectation branch is `ExpectationNLP`, a different type with a different solver.
- `LikelihoodNLP` → **`FullBatchNLP`**, the deterministic all-terms view. The old name
  survives as an alias; the new one says what it is.
- **`N_max` dropped from every sampling rule.** On a finite sum the cap is $M$, imposed by
  the problem, and a rule carrying a user `N_max` is now rejected at oracle construction —
  there it is either redundant ($\geq M$) or a deliberate sub-population budget ($< M$),
  and those are different intentions. Pass `budget` to the oracle for the second.

`BHHHModel` now requires a `LikelihoodProblem` and `GaussNewtonModel` an `NLSProblem`,
checked at solver construction. That check prevents a silent failure: applied to the wrong
problem BHHH still produces a positive semidefinite matrix and the run still converges,
and nothing in $\rho$, $\|g\|$ or the radius trace reveals that the model approximates
nothing in particular.

In [ ]:
bhhh_study()

In [ ]:
sampling_examples()